# Методы и технологии глубокого обучения: комплексное учебное пособие

Данное пособие синтезирует учебные материалы по архитектурам, методам обучения и метрикам оценки глубоких нейронных сетей.

Цель notebook - подготовить студента к пониманию нейросетей в контексте ВКР:

```text
classic ML classification -> CNN image classification -> transfer learning -> YOLO/object detection
```

В этом notebook формулы записаны в Markdown через LaTeX, чтобы они нормально читались в Jupyter и GitHub.

## 1. Deep Learning vs Classic Machine Learning

В классическом машинном обучении человек часто сам проектирует признаки.

Пример classic ML для животных:

```text
weight_kg, height_cm, wool_density -> RandomForest -> cattle/sheep
```

В глубоком обучении модель получает более сырые данные и сама строит иерархию признаков.

Пример DL для ВКР:

```text
image crop -> CNN/VGG16 -> cattle/sheep
```

Главное отличие:

| Classic ML | Deep Learning |
|---|---|
| Признаки часто задаются вручную | Признаки извлекаются автоматически |
| Хорошо работает на табличных данных | Сильно работает на изображениях, аудио, тексте |
| Обычно меньше данных и вычислений | Обычно больше данных и вычислений |
| Модели легче интерпретировать | Модели сложнее интерпретировать |

## 2. Искусственный нейрон

Искусственный нейрон получает входные признаки $x_1, x_2, \dots, x_n$.

Каждый вход умножается на обучаемый вес $w_i$. Затем добавляется смещение $b$.

Сначала считается линейная комбинация:

$$
z = \sum_{i=1}^{n} w_i x_i + b
$$

Затем применяется функция активации:

$$
y = f(z)
$$

Итоговая формула нейрона:

$$
y = f\left(\sum_{i=1}^{n} w_i x_i + b\right)
$$

Где:

- $x_i$ - входные данные или признаки;
- $w_i$ - веса;
- $b$ - bias / смещение;
- $f$ - функция активации;
- $y$ - выход нейрона.

In [ ]:
# Мини-пример искусственного нейрона на Python без deep learning frameworks.

# Входные признаки: например, два признака животного.
x1 = 0.8  # нормализованный вес
x2 = 0.3  # нормализованная плотность шерсти

# Веса нейрона.
w1 = 2.0
w2 = -1.0

# Bias / смещение.
b = 0.1

# Линейная часть нейрона: z = w1*x1 + w2*x2 + b.
z = w1 * x1 + w2 * x2 + b

# ReLU activation: f(z) = max(0, z).
y = max(0, z)

print("z =", z)
print("y = ReLU(z) =", y)

## 3. Функции активации

Без функции активации нейросеть была бы просто набором линейных преобразований.

Нелинейность нужна, чтобы сеть могла моделировать сложные зависимости.

### Sigmoid

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

Sigmoid сжимает значение в диапазон $(0, 1)$, но может страдать от затухающих градиентов.

### Tanh

$$
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}
$$

Tanh сжимает значение в диапазон $(-1, 1)$.

### ReLU

$$
f(x) = \max(0, x)
$$

ReLU стала стандартом во многих глубоких сетях, потому что:

- вычисляется очень просто;
- сохраняет градиент для положительных значений;
- уменьшает проблему vanishing gradients по сравнению с sigmoid/tanh.

In [ ]:
# Визуализация функций активации.
import numpy as np
import matplotlib.pyplot as plt

# Создаем значения x от -6 до 6.
x = np.linspace(-6, 6, 400)

# Считаем sigmoid.
sigmoid = 1 / (1 + np.exp(-x))

# Считаем tanh.
tanh = np.tanh(x)

# Считаем ReLU.
relu = np.maximum(0, x)

# Создаем график.
plt.figure(figsize=(10, 5))

# Рисуем все три функции.
plt.plot(x, sigmoid, label="Sigmoid")
plt.plot(x, tanh, label="Tanh")
plt.plot(x, relu, label="ReLU")

# Добавляем горизонтальную линию y=0.
plt.axhline(0, color="black", linewidth=0.8)

# Добавляем вертикальную линию x=0.
plt.axvline(0, color="black", linewidth=0.8)

# Подписываем график.
plt.title("Activation functions")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True)
plt.show()

## 4. Сверточная нейронная сеть (CNN)

CNN используется для изображений.

Главная идея: вместо ручного задания признаков модель сама учит фильтры, которые находят визуальные паттерны.

### Слой свертки

Свертка применяет маленькое ядро / фильтр к участкам изображения.

Упрощенно для одного положения:

$$
S(i,j) = \sum_m \sum_n I(i+m, j+n)K(m,n)
$$

Где:

- $I$ - входное изображение или карта признаков;
- $K$ - ядро свертки;
- $S$ - новая карта признаков.

Ранние слои могут находить простые признаки:

- края;
- линии;
- углы;
- текстуры.

Более глубокие слои находят более абстрактные признаки:

- форму тела;
- морду;
- шерсть;
- целый объект.

## 5. Pooling: Max Pooling и Average Pooling

Pooling уменьшает spatial размер карты признаков.

Это нужно, чтобы:

- снизить вычислительные затраты;
- сделать признаки более устойчивыми к небольшим сдвигам;
- оставить наиболее важную информацию.

### Max Pooling

Берет максимум в окне:

$$
\text{MaxPool}(R) = \max_{x \in R} x
$$

Max Pooling выделяет самый сильный признак в окне.

### Average Pooling

Берет среднее значение:

$$
\text{AvgPool}(R) = \frac{1}{|R|}\sum_{x \in R} x
$$

Average Pooling сглаживает информацию.

Коротко:

| Метод | Что делает | Когда полезен |
|---|---|---|
| Max Pooling | Берет максимум | Выделить доминирующий признак |
| Average Pooling | Берет среднее | Сгладить карту признаков |

In [ ]:
# Мини-пример max pooling и average pooling на маленькой матрице 2x2.
import numpy as np

# Представим, что это маленький участок feature map.
region = np.array([
    [1, 3],
    [2, 8]
])

# Max pooling берет самое большое число.
max_pool = region.max()

# Average pooling берет среднее.
avg_pool = region.mean()

print("Region:")
print(region)
print("Max Pooling:", max_pool)
print("Average Pooling:", avg_pool)

## 6. Fully Connected Layer

Полносвязный слой обычно находится ближе к концу CNN.

Его задача - взять признаки, извлеченные convolution/pooling слоями, и принять финальное решение.

Для классификации он часто выдает logits для каждого класса:

```text
features -> fully connected layer -> logits -> softmax -> probabilities
```

Если классы `cattle` и `sheep`, то модель может выдать два logits:

$$
\mathbf{z} = [z_{cattle}, z_{sheep}]
$$

Затем softmax превращает logits в вероятности:

$$
p_i = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}
$$

Где $C$ - количество классов.

## 7. Loss function и обучение

Модель обучается, минимизируя функцию потерь.

Для многоклассовой классификации часто используется cross-entropy loss:

$$
L = -\sum_{i=1}^{C} y_i \log(\hat{y}_i)
$$

Где:

- $C$ - количество классов;
- $y_i$ - истинная метка в one-hot формате;
- $\hat{y}_i$ - предсказанная вероятность класса.

Если модель уверенно предсказывает правильный класс, loss маленький.

Если модель уверенно ошибается, loss большой.

## 8. Backpropagation

Backpropagation - механизм обучения нейросети.

Идея:

1. Сделать forward pass: получить prediction.
2. Посчитать loss.
3. Посчитать градиенты loss по весам.
4. Обновить веса так, чтобы loss уменьшался.

Обновление весов в gradient descent:

$$
w_{t+1} = w_t - \eta \frac{\partial L}{\partial w_t}
$$

Где:

- $w_t$ - вес на текущем шаге;
- $w_{t+1}$ - вес после обновления;
- $\eta$ - learning rate;
- $L$ - функция потерь;
- $\frac{\partial L}{\partial w_t}$ - градиент.

Backpropagation идет от выхода к входу, поэтому называется обратным распространением ошибки.

## 9. Vanishing gradients и ResNet

В очень глубоких сетях градиенты могут становиться слишком маленькими при движении назад через слои.

Это называется проблемой исчезающего градиента.

Если градиент почти нулевой, ранние слои почти не обучаются.

### Остаточные связи / skip connections

ResNet добавляет shortcut path:

$$
y = F(x) + x
$$

Где:

- $x$ - вход блока;
- $F(x)$ - преобразование через несколько слоев;
- $F(x) + x$ - выход блока.

Идея: сеть учит не полное преобразование, а остаток / residual.

Это помогает обучать очень глубокие модели, например ResNet-50, ResNet-101, ResNet-152.

## 10. Регуляризация: борьба с переобучением

Переобучение возникает, когда модель слишком хорошо запоминает train data, но плохо работает на новых данных.

Методы регуляризации:

### Dropout

Во время обучения случайно выключает часть нейронов.

Если вероятность dropout равна $p$, то каждый нейрон может быть выключен с вероятностью $p$.

Это заставляет сеть не полагаться на один конкретный путь.

### Batch Normalization

Нормализует активации внутри сети:

$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$

Где:

- $\mu$ - среднее;
- $\sigma^2$ - дисперсия;
- $\epsilon$ - маленькое число для численной стабильности.

BatchNorm стабилизирует и часто ускоряет обучение.

### Early Stopping

Останавливает обучение, если validation loss перестает улучшаться.

## 11. Transfer Learning

Transfer Learning - использование модели, предобученной на большом dataset, для новой задачи.

Пример:

```text
ImageNet pretrained VGG16 -> cattle/sheep classifier
```

Почему это эффективно:

- ранние слои уже умеют находить края, текстуры, формы;
- нужно меньше данных;
- обучение быстрее;
- качество часто выше, чем обучение с нуля.

Два режима:

| Режим | Что обучается |
|---|---|
| Feature extraction | Backbone frozen, обучается только head |
| Fine-tuning | Размораживается часть backbone и head |

В ВКР transfer learning использовался для ResNet50, VGG16 и MobileNetV2.

## 12. Метрики classification

Для классификации недостаточно одной accuracy, особенно если классы несбалансированы.

### Accuracy

$$
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
$$

### Precision

$$
Precision = \frac{TP}{TP + FP}
$$

Precision отвечает: из всех объектов, которые модель назвала positive, сколько действительно positive?

### Recall

$$
Recall = \frac{TP}{TP + FN}
$$

Recall отвечает: из всех реальных positive объектов, сколько модель нашла?

### F1-score

$$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

F1 балансирует precision и recall.

## 13. Метрики object detection: IoU и mAP

Для детекции объектов нужно оценивать не только класс, но и качество bounding box.

### IoU

IoU / Intersection over Union:

$$
IoU = \frac{Area(B_{pred} \cap B_{true})}{Area(B_{pred} \cup B_{true})}
$$

Где:

- $B_{pred}$ - predicted bounding box;
- $B_{true}$ - ground truth bounding box.

Интерпретация:

- $IoU = 1$ - идеальное совпадение;
- $IoU = 0$ - нет пересечения;
- часто detection считается корректным при $IoU \ge 0.5$.

### mAP

mAP / Mean Average Precision - средняя precision по классам и confidence thresholds.

Для YOLO обычно используют:

- `mAP@50`;
- `mAP@50-95`.

## 14. Тест для проверки знаний

Инструкция: ответьте на вопросы. Каждый ответ должен состоять из 2-3 предложений.

1. В чем принципиальное отличие глубокого обучения (DL) от классического машинного обучения (ML)?
2. Опишите математическую основу работы искусственного нейрона.
3. Что такое остаточные связи / skip connections и какую проблему они решают в ResNet?
4. Какую роль выполняет fully connected layer в структуре CNN?
5. Объясните разницу между Max Pooling и Average Pooling.
6. Что понимается под скрытыми признаками в контексте глубокого обучения?
7. Почему ReLU стала стандартом в современных глубоких сетях по сравнению с sigmoid/tanh?
8. В чем суть backpropagation?
9. Что такое Transfer Learning и почему этот метод эффективен в инженерных задачах?
10. Для чего используется IoU и как она интерпретируется?

## 15. Ответы на тестовые вопросы

### 1. Отличие DL от ML

В классическом машинном обучении признаки часто задаются экспертами вручную. В глубоком обучении модель автоматически извлекает иерархические признаки напрямую из сырых данных, например изображений или сигналов.

### 2. Основа нейрона

Нейрон вычисляет взвешенную сумму входов с добавлением смещения:

$$
z = \sum_{i=1}^{n} w_i x_i + b
$$

Затем применяется нелинейная функция активации:

$$
y = f(z)
$$

### 3. Skip connections в ResNet

Skip connections позволяют сигналу проходить через блок напрямую. В ResNet это записывается как $y = F(x) + x$, что помогает бороться с исчезающими градиентами и обучать очень глубокие сети.

### 4. Fully connected layer

Fully connected layer обычно находится в конце CNN и принимает решение на основе признаков, извлеченных сверточными слоями. Для классификации он преобразует признаки в logits по классам.

### 5. Max Pooling vs Average Pooling

Max Pooling выбирает максимум в окне и выделяет наиболее сильный признак. Average Pooling берет среднее значение и больше сглаживает карту признаков.

### 6. Скрытые признаки

Скрытые признаки - это внутренние абстрактные представления, которые сеть строит сама. В изображениях это может быть переход от краев и текстур к частям объекта и затем к целому объекту.

### 7. Почему ReLU

ReLU задается как $f(x)=\max(0,x)$ и вычисляется очень быстро. Для положительных значений она сохраняет градиент, поэтому меньше страдает от vanishing gradients, чем sigmoid и tanh.

### 8. Backpropagation

Backpropagation вычисляет градиенты функции потерь по всем весам сети от выхода к входу. Эти градиенты используются оптимизатором, чтобы обновить веса и уменьшить ошибку.

### 9. Transfer Learning

Transfer Learning использует веса модели, предобученной на большом dataset, например ImageNet. Это эффективно, потому что модель уже знает общие визуальные признаки и быстрее адаптируется к новой задаче.

### 10. IoU

IoU оценивает степень пересечения predicted bounding box и ground truth bounding box:

$$
IoU = \frac{Area(B_{pred} \cap B_{true})}{Area(B_{pred} \cup B_{true})}
$$

Значение 1 означает идеальное совпадение, а 0 - отсутствие пересечения.

## 16. Темы для эссе

1. Эволюция архитектур CNN: от LeNet и AlexNet до ResNet и EfficientNet.
2. Влияние регуляризации на обобщающую способность: Dropout, Batch Normalization, Early Stopping.
3. Transfer Learning в инженерных задачах: почему предобученные модели полезны при малом количестве данных.
4. Проблема исчезающих градиентов и методы ее преодоления в глубоких сетях.
5. Метрики качества в computer vision: почему accuracy недостаточно для segmentation и detection.
6. Связь classic ML, CNN и YOLO: как меняется постановка задачи от табличной классификации к object detection.

## 17. Глоссарий ключевых терминов

| Термин | Определение |
|---|---|
| Автоэнкодер | Алгоритм обучения без учителя, который сжимает данные и затем восстанавливает входной сигнал. |
| Аугментация данных | Искусственное расширение обучающей выборки через повороты, сдвиги, масштабирование, шум и другие трансформации. |
| Глубина сети | Количество слоев нейронной сети. |
| Дропаут / Dropout | Метод регуляризации, при котором во время обучения случайно выключается часть нейронов. |
| Инвариантность | Способность модели сохранять правильное распознавание при небольших сдвигах, поворотах или изменениях масштаба. |
| Карта признаков | Матрица активаций после свертки, показывающая наличие признаков в разных местах изображения. |
| Компьютерное зрение | Раздел AI, который извлекает информацию из изображений и видео. |
| Batch Normalization | Метод нормализации активаций между слоями для стабилизации и ускорения обучения. |
| Оптимизатор | Алгоритм обновления весов, например SGD или Adam. |
| Слой свертки | Основной слой CNN, применяющий обучаемые фильтры к изображению или feature map. |
| Pooling | Операция уменьшения spatial размера feature map. |
| Функция активации | Нелинейная функция, позволяющая сети моделировать сложные зависимости. |
| Ширина сети | Количество нейронов или каналов в слое. |
| Ядро свертки / фильтр | Маленькая матрица весов, которая перемещается по входу и ищет паттерны. |
| F1-score | Гармоническое среднее precision и recall. |
| IoU | Метрика пересечения predicted и ground truth области. |
| mAP | Mean Average Precision, основная метрика качества object detection. |
| Transfer Learning | Использование предобученной модели для новой задачи. |
| Fine-tuning | Дообучение части или всей предобученной модели на новом dataset. |
| Feature extraction | Режим transfer learning, где backbone frozen, а обучается только новая head. |

## 18. Связь с ВКР

В ВКР используются те же базовые идеи:

### Custom CNN baseline

Собственная CNN учит признаки из crop-изображений cattle/sheep с нуля.

### Transfer Learning

ResNet50, VGG16 и MobileNetV2 используют предобученные ImageNet features.

### Grad-CAM

Grad-CAM помогает проверить, смотрит ли модель на тело животного или на фон.

### YOLOv8

YOLOv8 нужна для следующего этапа: найти всех животных на полном изображении и посчитать их количество.

Итоговая цепочка понимания:

```text
classic ML: human-designed tabular features -> class
CNN: pixels -> learned visual features -> class
YOLO: full image -> boxes + classes -> count
```